In [0]:
%run "../SetUp/setup" 

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


In [0]:
%run "../Includes/configs" 

In [0]:
%run "../Includes/comm_func" 

**Produce Driver Standings**

In [0]:
dbutils.widgets.text("p_file_date", "")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
#race_results_list = spark.read.parquet(f"{presentation_folder_path}/race_results").filter(f"file_date = '{v_file_date}'").select("race_year").distinct().collect()

In [0]:
#race_year_list = []
#for race_year in race_results_list:
#    race_year_list.append(race_year.race_year)
#print(race_year_list) 

In [0]:
#from pyspark.sql.functions import col

In [0]:
#race_results_df = spark.read.parquet(f"{presentation_folder_path}/race_results").filter(col("race_year").isin(race_year_list))

In [0]:
from pyspark.sql.functions import sum, when, col, count, lit

In [0]:
driver_standings_df = race_results_df.groupBy("race_year", "driver_name", "driver_nationality", "team").agg(sum("points").alias("total_points"), count(when(col("position") == 1, True)).alias("wins"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc, asc

In [0]:
drivers_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"), desc("wins"))
final_df = driver_standings_df.withColumn("rank", rank().over(drivers_rank_spec)).withColumn("file_date", lit(v_file_date))

In [0]:
#final_df.write.mode("overwrite").parquet(f"{presentation_folder_path}/driver_standings")

In [0]:
final_df_dedup = final_df.dropDuplicates(
    ["race_year", "driver_name"]
)

In [0]:
final_df_dedup.write \
    .mode("append") \
    .partitionBy("race_year") \
    .parquet(f"{presentation_folder_path}/driver_standings")

In [0]:
spark.read.parquet(f"{presentation_folder_path}/driver_standings") \
    .select("file_date") \
    .distinct() \
    .show(truncate=False)

+----------+
|file_date |
+----------+
|2021-04-18|
|2021-03-28|
|2021-03-21|
+----------+



In [0]:
final_df.count()

20

In [0]:
%sql
SELECT race_year, driver_name
FROM f1_presentation.driver_standings;

race_year,driver_name
2021,Lewis Hamilton
2021,Max Verstappen
2021,Lando Norris
2021,Charles Leclerc
2021,Valtteri Bottas
2021,Carlos Sainz
2021,Daniel Ricciardo
2021,Sergio Pérez
2021,Pierre Gasly
2021,Lance Stroll


In [0]:
%sql
CREATE TABLE IF NOT EXISTS f1_presentation.driver_standings (
  race_year INT,
  driver_name STRING,
  driver_nationality STRING,
  team STRING,
  total_points DOUBLE,
  wins BIGINT,
  rank INT,
  file_date DATE
)
USING DELTA
PARTITIONED BY (race_year);

In [0]:
merge_into_table(
    final_df,
    "f1_presentation",
    "driver_standings",
    """
    target.race_year = source.race_year
    AND target.driver_name = source.driver_name
    AND target.file_date = source.file_date
    """
)

In [0]:
%sql
SELECT * from f1_presentation.driver_standings;

race_year,driver_name,driver_nationality,team,total_points,wins,rank,file_date
2021,Lewis Hamilton,British,Mercedes,44.0,1,1,2021-03-28
2021,Max Verstappen,Dutch,Red Bull,43.0,1,2,2021-03-28
2021,Lando Norris,British,McLaren,27.0,0,3,2021-03-28
2021,Charles Leclerc,Monegasque,Ferrari,20.0,0,4,2021-03-28
2021,Valtteri Bottas,Finnish,Mercedes,16.0,0,5,2021-03-28
2021,Carlos Sainz,Spanish,Ferrari,14.0,0,6,2021-03-28
2021,Daniel Ricciardo,Australian,McLaren,14.0,0,6,2021-03-28
2021,Sergio Pérez,Mexican,Red Bull,10.0,0,8,2021-03-28
2021,Pierre Gasly,French,AlphaTauri,6.0,0,9,2021-03-28
2021,Lance Stroll,Canadian,Aston Martin,5.0,0,10,2021-03-28
